# COSTAR-TS Results Visualization

This notebook visualizes the consolidated ETTh1 and ETTh2 COSTAR result table from `all_costar_results.csv`.

It focuses on rows with test metrics, while keeping labels for clean preregistered results, pre-test frozen audits, validation-tuned rows, and after-final-test audits.

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    sns = None
    plt.style.use("default")

ROOT = Path.cwd()
if not (ROOT / "experiments" / "all_results_summary" / "all_costar_results.csv").exists():
    ROOT = Path.cwd().parents[0]

CSV_PATH = ROOT / "experiments" / "all_results_summary" / "all_costar_results.csv"
df = pd.read_csv(CSV_PATH)

for col in ["test_mae", "test_mse", "validation_mae", "validation_mse", "diff_vs_anchor", "diff_vs_validation"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

test = df[df["test_mae"].notna()].copy()
test["method_display"] = test["method"].astype(str).str.replace("_", " ", regex=False)
test["status_display"] = test["status"].fillna("unknown").astype(str)

print(f"Loaded {len(df)} total rows from {CSV_PATH}")
print(f"Rows with test MAE: {len(test)}")
display(test[["dataset", "method", "test_mae", "test_mse", "validation_mae", "status", "result_group"]].sort_values(["dataset", "test_mae"]))

## Helpers

In [ ]:
STATUS_COLORS = {
    "clean_preregistered": "#1f77b4",
    "pre_test_frozen": "#2ca02c",
    "validation_selected_reference": "#9467bd",
    "after_final_test_audit": "#ff7f0e",
    "etth2_validation_tuned": "#d62728",
    "locked_etth1_config_etth2_replication": "#8c564b",
}

def color_for_status(status):
    return STATUS_COLORS.get(str(status), "#7f7f7f")

def short_label(text, width=36):
    return "\n".join(textwrap.wrap(str(text).replace("_", " "), width=width))

def plot_ranked_bars(data, dataset, metric="test_mae", top_n=None):
    sub = data[data["dataset"] == dataset].sort_values(metric, ascending=True).copy()
    if top_n is not None:
        sub = sub.head(top_n)
    sub = sub.iloc[::-1]
    fig_h = max(5, 0.42 * len(sub))
    fig, ax = plt.subplots(figsize=(11, fig_h))
    colors = [color_for_status(s) for s in sub["status_display"]]
    ax.barh(range(len(sub)), sub[metric], color=colors, alpha=0.88)
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels([short_label(x) for x in sub["method"]])
    ax.set_xlabel(metric.replace("_", " ").upper())
    ax.set_title(f"{dataset} ranked by {metric.replace('_', ' ')}")
    ax.grid(axis="x", alpha=0.25)
    for i, value in enumerate(sub[metric]):
        ax.text(value, i, f" {value:.6f}", va="center", fontsize=9)
    return fig, ax

def status_legend(ax):
    from matplotlib.patches import Patch
    handles = [Patch(color=color, label=status) for status, color in STATUS_COLORS.items() if status in set(test["status_display"])]
    ax.legend(handles=handles, loc="best", frameon=True, fontsize=8)

## ETTh1 Ranking

In [ ]:
fig, ax = plot_ranked_bars(test, "ETTh1", metric="test_mae")
status_legend(ax)
plt.tight_layout()
plt.show()

## ETTh2 Ranking

In [ ]:
fig, ax = plot_ranked_bars(test, "ETTh2", metric="test_mae")
status_legend(ax)
plt.tight_layout()
plt.show()

## Validation vs Test

Lower-left is better. Points far from the diagonal indicate validation/test shift.

In [ ]:
plot_df = test[test["validation_mae"].notna()].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, dataset in zip(axes, ["ETTh1", "ETTh2"]):
    sub = plot_df[plot_df["dataset"] == dataset]
    for status, group in sub.groupby("status_display"):
        ax.scatter(group["validation_mae"], group["test_mae"], s=70, alpha=0.85, label=status, color=color_for_status(status))
    for _, row in sub.iterrows():
        label = str(row["method"]).replace("nonnegative_simplex_linear_average", "simplex")
        label = label.replace("Ridge residual corrector", "ridge resid")
        label = label.replace("MLP residual corrector", "MLP resid")
        if row["test_mae"] <= sub["test_mae"].quantile(0.30) or "simplex" in label:
            ax.annotate(label[:28], (row["validation_mae"], row["test_mae"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_title(dataset)
    ax.set_xlabel("Validation MAE")
    ax.set_ylabel("Test MAE")
    ax.grid(alpha=0.25)

handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles, labels, fontsize=8, loc="best")
plt.tight_layout()
plt.show()

## Gap To Official Full Adaptive Model

Negative means the method had lower test MAE than the official clean full adaptive result for that dataset.

In [ ]:
official = {
    "ETTh1": 0.3263952910900116,
    "ETTh2": 0.29780814051628113,
}

gap = test.copy()
gap["diff_vs_official_full_adaptive"] = gap.apply(lambda r: r["test_mae"] - official.get(r["dataset"], np.nan), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=False)
for ax, dataset in zip(axes, ["ETTh1", "ETTh2"]):
    sub = gap[gap["dataset"] == dataset].sort_values("diff_vs_official_full_adaptive").copy()
    sub = sub.iloc[::-1]
    colors = ["#2ca02c" if x < 0 else "#d62728" if x > 0 else "#7f7f7f" for x in sub["diff_vs_official_full_adaptive"]]
    ax.barh(range(len(sub)), sub["diff_vs_official_full_adaptive"], color=colors, alpha=0.85)
    ax.axvline(0, color="black", lw=1)
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels([short_label(x, 28) for x in sub["method"]], fontsize=8)
    ax.set_title(dataset)
    ax.set_xlabel("Test MAE minus official full adaptive")
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

display(gap[["dataset", "method", "test_mae", "diff_vs_official_full_adaptive", "status"]].sort_values(["dataset", "diff_vs_official_full_adaptive"]))

## Simplex And Linear/Residual Methods

In [ ]:
keywords = "simplex|ridge|MLP|residual|linear stacker|Full frozen adaptive|Full adaptive|fixed core|Best single"
focus = test[test["method"].astype(str).str.contains(keywords, case=False, regex=True, na=False)].copy()
focus = focus.sort_values(["dataset", "test_mae"])
display(focus[["dataset", "method", "test_mae", "test_mse", "validation_mae", "validation_mse", "status", "result_group"]])

fig, ax = plt.subplots(figsize=(12, 6))
datasets = ["ETTh1", "ETTh2"]
xbase = np.arange(len(datasets))
for i, dataset in enumerate(datasets):
    sub = focus[focus["dataset"] == dataset].sort_values("test_mae").head(8)
    offsets = np.linspace(-0.32, 0.32, len(sub)) if len(sub) else []
    for off, (_, row) in zip(offsets, sub.iterrows()):
        ax.scatter(i + off, row["test_mae"], s=90, color=color_for_status(row["status_display"]), alpha=0.9)
        ax.text(i + off, row["test_mae"], short_label(row["method"], 16), rotation=65, fontsize=8, ha="left", va="bottom")
ax.set_xticks(xbase)
ax.set_xticklabels(datasets)
ax.set_ylabel("Test MAE")
ax.set_title("Focused Comparison: Simplex, Linear Stackers, Residuals, Full Adaptive")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Export Clean Ranked Tables

In [ ]:
out_dir = CSV_PATH.parent
ranked = test.sort_values(["dataset", "test_mae"])[[
    "dataset", "method", "expert_set", "test_mae", "test_mse", "validation_mae", "validation_mse", "status", "result_group", "selection_protocol", "source_file"
]].copy()
ranked["rank_within_dataset"] = ranked.groupby("dataset")["test_mae"].rank(method="first", ascending=True).astype(int)
ranked = ranked[["rank_within_dataset", *[c for c in ranked.columns if c != "rank_within_dataset"]]]
export_path = out_dir / "ranked_test_results_for_visualization.csv"
ranked.to_csv(export_path, index=False)
print(f"Wrote {export_path}")
display(ranked)